# Notebook 08: Model Quantization & Optimization for ESP32-S3

**Purpose:** Convert and optimize the best neural model for embedded deployment

**Objectives:**
1. Load best model from notebook 07
2. Convert to TensorFlow Lite format
3. Apply INT8 quantization (representative dataset)
4. Validate accuracy post-quantization
5. Benchmark inference speed and memory
6. Generate deployment package for ESP32-S3

**Target Device: ESP32-S3**
- Processor: Xtensa dual-core LX7 @ 240 MHz
- SRAM: ~512 KB available
- Flash: 8-16 MB (model storage)
- Target model size: <500 KB
- Target inference: <50 ms per sample

**Quantization Benefits:**
- INT8: ~4x smaller than FP32
- Faster inference on embedded devices
- Lower power consumption
- Typical accuracy loss: 1-2%

---

## Section 1: Setup & Configuration

In [ ]:
import os, sys
from pathlib import Path
import json
import time
from datetime import datetime
import warnings
import gc
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

# TensorFlow/Keras
import tensorflow as tf
from tensorflow import keras

# Sklearn
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report, roc_auc_score
)

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

print(f'TensorFlow version: {tf.__version__}')
print(f'Keras version: {keras.__version__}')
print('✓ All libraries imported')

In [ ]:
# Project paths
PROJECT_ROOT = Path.cwd().parent
FEATURES_DIR = PROJECT_ROOT / 'data' / 'features'
MODELS_DIR = PROJECT_ROOT / 'models' / 'neural'
TFLITE_DIR = PROJECT_ROOT / 'models' / 'tflite'
RESULTS_DIR = PROJECT_ROOT / 'results'
FIGURES_DIR = RESULTS_DIR / 'figures'

# Create directories
TFLITE_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root:  {PROJECT_ROOT}')
print(f'Models dir:    {MODELS_DIR}')
print(f'TFLite dir:    {TFLITE_DIR}')
print(f'Results dir:   {RESULTS_DIR}')

In [ ]:
# Set random seed
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f'✓ Random seed set to {SEED}')

## Section 2: Load Best Model from Notebook 07

In [ ]:
print('\nLOADING BEST MODEL FROM NOTEBOOK 07...')
print('='*70)

# Check for available models
model_1ch = MODELS_DIR / 'best_neural_model_1ch.keras'
model_3ch = MODELS_DIR / 'best_neural_model_3ch.keras'

print('\nAvailable models:')
print(f'  1-channel: {"✓" if model_1ch.exists() else "✗"} {model_1ch.name}')
print(f'  3-channel: {"✓" if model_3ch.exists() else "✗"} {model_3ch.name}')

# Choose which model to quantize
USE_3CH = True  # Set to False to quantize 1ch model

if USE_3CH and model_3ch.exists():
    model_path = model_3ch
    feature_type = '3ch'
    report_file = RESULTS_DIR / '07_neural_recommendation_report_3ch.json'
elif model_1ch.exists():
    model_path = model_1ch
    feature_type = '1ch'
    report_file = RESULTS_DIR / '07_neural_recommendation_report_1ch.json'
else:
    raise FileNotFoundError('No trained models found. Run notebook 07 first.')

print(f'\nSelected model: {feature_type}')

# Load model
print(f'Loading {model_path.name}...')
model = keras.models.load_model(model_path)

print(f'✓ Model loaded successfully')
print(f'  Architecture: {model.name}')
print(f'  Parameters:   {model.count_params():,}')
print(f'  Input shape:  {model.input_shape}')
print(f'  Output shape: {model.output_shape}')

# Load report
if report_file.exists():
    with open(report_file, 'r') as f:
        report = json.load(f)
    
    print(f'\nPre-quantization metrics (from notebook 07):')
    print(f'  Model:     {report["best_neural_model"]["model_name"]}')
    print(f'  F1:        {report["best_neural_model"]["test_f1"]:.4f}')
    print(f'  Recall:    {report["best_neural_model"]["recall"]:.4f}')
    print(f'  Inference: {report["best_neural_model"]["inference_ms"]:.2f} ms')
    print(f'  Size:      {report["best_neural_model"]["size_mb"]:.2f} MB')
    
    original_f1 = report['best_neural_model']['test_f1']
    original_recall = report['best_neural_model']['recall']
    original_size_mb = report['best_neural_model']['size_mb']
else:
    print('⚠️  Report file not found - will skip comparison')
    original_f1 = None
    original_recall = None
    original_size_mb = None

print('='*70)

## Section 3: Load Test Data

In [ ]:
print('\nLOADING TEST DATA...')
print('='*70)

# Load spectrograms
if feature_type == '3ch':
    spec_file = FEATURES_DIR / 'spectrograms_3ch.npy'
else:
    spec_file = FEATURES_DIR / 'spectrograms_1ch.npy'

labels_file = FEATURES_DIR / 'labels.npy'

print(f'Loading {spec_file.name}...')
X_spec = np.load(spec_file)
y = np.load(labels_file)

print(f'✓ Data loaded')
print(f'  Shape:   {X_spec.shape}')
print(f'  Labels:  {y.shape}')
print(f'  Memory:  {X_spec.nbytes / 1024**2:.1f} MB')

# Create same train/test split as notebook 07
from sklearn.model_selection import train_test_split

X_temp, X_test, y_temp, y_test = train_test_split(
    X_spec, y, test_size=0.15, random_state=SEED, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.176, random_state=SEED, stratify=y_temp
)

print(f'\nData splits:')
print(f'  Train: {X_train.shape[0]:,} samples')
print(f'  Val:   {X_val.shape[0]:,} samples')
print(f'  Test:  {X_test.shape[0]:,} samples')

# Normalize same as notebook 07
def norm_sample(x):
    mean = x.mean()
    std = x.std() + 1e-8
    return (x - mean) / std

print('\nNormalizing data...')
X_train_norm = np.zeros_like(X_train, dtype=np.float32)
X_val_norm   = np.zeros_like(X_val,   dtype=np.float32)
X_test_norm  = np.zeros_like(X_test,  dtype=np.float32)

for i in range(len(X_train)):
    X_train_norm[i] = norm_sample(X_train[i])

for i in range(len(X_val)):
    X_val_norm[i] = norm_sample(X_val[i])

for i in range(len(X_test)):
    X_test_norm[i] = norm_sample(X_test[i])

print('✓ Normalization complete')

# Prepare input based on model architecture
if len(X_test_norm.shape) == 3:
    X_test_final = X_test_norm
    X_train_final = X_train_norm
elif len(X_test_norm.shape) == 4:
    # Reshape for CNN/LSTM: (samples, time, freq*channels)
    N_test, T, F, C = X_test_norm.shape
    N_train, _, _, _ = X_train_norm.shape
    X_test_final = X_test_norm.reshape(N_test, T, F * C)
    X_train_final = X_train_norm.reshape(N_train, T, F * C)
    print(f'Reshaped 4D to 3D for CNN/LSTM: {X_test_final.shape}')

# For MLP, may need to flatten
if 'MLP' in model.name or 'mlp' in model.name.lower():
    # MLP will flatten internally, keep as-is
    pass

# Memory cleanup
del X_spec, X_temp, X_train, X_val, X_test
del X_train_norm, X_val_norm, X_test_norm
gc.collect()

print('\n✓ Test data ready')
print(f'  Final shape: {X_test_final.shape}')
print('='*70)

## Section 4: Baseline Keras Model Evaluation

In [ ]:
print('\nBASELINE KERAS MODEL EVALUATION...')
print('='*70)

# Predictions
start_time = time.time()
y_pred_keras = model.predict(X_test_final, verbose=0)
keras_inference_time = (time.time() - start_time) / len(X_test_final) * 1000

y_pred_keras_binary = (y_pred_keras > 0.5).astype(int).flatten()

# Metrics
keras_acc = accuracy_score(y_test, y_pred_keras_binary)
keras_f1 = f1_score(y_test, y_pred_keras_binary)
keras_precision = precision_score(y_test, y_pred_keras_binary)
keras_recall = recall_score(y_test, y_pred_keras_binary)
keras_auc = roc_auc_score(y_test, y_pred_keras)

print('Baseline Keras Model (FP32):')
print(f'  Accuracy:  {keras_acc:.4f}')
print(f'  F1 Score:  {keras_f1:.4f}')
print(f'  Precision: {keras_precision:.4f}')
print(f'  Recall:    {keras_recall:.4f}')
print(f'  AUC:       {keras_auc:.4f}')
print(f'  FNR:       {(1 - keras_recall) * 100:.2f}%')
print(f'  Inference: {keras_inference_time:.2f} ms/sample')

# Model size
keras_size_mb = model_path.stat().st_size / (1024 * 1024)
print(f'  Size:      {keras_size_mb:.2f} MB ({keras_size_mb * 1024:.0f} KB)')

print('='*70)

## Section 5: Convert to TensorFlow Lite (FP32 baseline)

In [ ]:
print('\nCONVERTING TO TENSORFLOW LITE (FP32)...')
print('='*70)

# Convert to TFLite (FP32 - no quantization yet)
converter = tf.lite.TFLiteConverter.from_keras_model(model)

# No optimizations for baseline
tflite_model_fp32 = converter.convert()

# Save
tflite_fp32_path = TFLITE_DIR / f'model_{feature_type}_fp32.tflite'
with open(tflite_fp32_path, 'wb') as f:
    f.write(tflite_model_fp32)

tflite_fp32_size_mb = len(tflite_model_fp32) / (1024 * 1024)

print(f'✓ TFLite FP32 model saved: {tflite_fp32_path.name}')
print(f'  Size: {tflite_fp32_size_mb:.2f} MB ({len(tflite_model_fp32) / 1024:.0f} KB)')
print(f'  Compression vs Keras: {(1 - tflite_fp32_size_mb/keras_size_mb) * 100:.1f}%')

print('='*70)

## Section 6: Evaluate TFLite FP32 Model

In [ ]:
print('\nEVALUATING TFLITE FP32 MODEL...')
print('='*70)

# Load TFLite model
interpreter_fp32 = tf.lite.Interpreter(model_path=str(tflite_fp32_path))
interpreter_fp32.allocate_tensors()

# Get input and output details
input_details = interpreter_fp32.get_input_details()
output_details = interpreter_fp32.get_output_details()

print('TFLite model details:')
print(f'  Input shape:  {input_details[0]["shape"]}')
print(f'  Input dtype:  {input_details[0]["dtype"]}')
print(f'  Output shape: {output_details[0]["shape"]}')
print(f'  Output dtype: {output_details[0]["dtype"]}')

# Run inference on test set
print('\nRunning inference on test set...')
y_pred_tflite_fp32 = []
inference_times = []

for i in range(len(X_test_final)):
    # Prepare input
    input_data = X_test_final[i:i+1].astype(np.float32)
    
    # Set input tensor
    interpreter_fp32.set_tensor(input_details[0]['index'], input_data)
    
    # Run inference
    start = time.time()
    interpreter_fp32.invoke()
    inference_times.append((time.time() - start) * 1000)
    
    # Get output
    output_data = interpreter_fp32.get_tensor(output_details[0]['index'])
    y_pred_tflite_fp32.append(output_data[0][0])

y_pred_tflite_fp32 = np.array(y_pred_tflite_fp32)
y_pred_tflite_fp32_binary = (y_pred_tflite_fp32 > 0.5).astype(int)

# Metrics
tflite_fp32_acc = accuracy_score(y_test, y_pred_tflite_fp32_binary)
tflite_fp32_f1 = f1_score(y_test, y_pred_tflite_fp32_binary)
tflite_fp32_precision = precision_score(y_test, y_pred_tflite_fp32_binary)
tflite_fp32_recall = recall_score(y_test, y_pred_tflite_fp32_binary)
tflite_fp32_auc = roc_auc_score(y_test, y_pred_tflite_fp32)
tflite_fp32_inference = np.mean(inference_times)

print('\nTFLite FP32 Model:')
print(f'  Accuracy:  {tflite_fp32_acc:.4f}')
print(f'  F1 Score:  {tflite_fp32_f1:.4f}')
print(f'  Precision: {tflite_fp32_precision:.4f}')
print(f'  Recall:    {tflite_fp32_recall:.4f}')
print(f'  AUC:       {tflite_fp32_auc:.4f}')
print(f'  FNR:       {(1 - tflite_fp32_recall) * 100:.2f}%')
print(f'  Inference: {tflite_fp32_inference:.2f} ms/sample (avg)')

# Compare with Keras
print('\nDifference vs Keras FP32:')
print(f'  F1:        {(tflite_fp32_f1 - keras_f1):+.4f}')
print(f'  Recall:    {(tflite_fp32_recall - keras_recall):+.4f}')
print(f'  Inference: {(tflite_fp32_inference - keras_inference_time):+.2f} ms')

print('='*70)

## Section 7: INT8 Quantization with Representative Dataset

In [ ]:
print('\nINT8 QUANTIZATION...')
print('='*70)

# Representative dataset generator
def representative_dataset_gen():
    """
    Generator for representative dataset.
    Uses training data to calibrate quantization.
    """
    # Use subset of training data (500 samples is usually enough)
    num_calibration_samples = min(500, len(X_train_final))
    
    for i in range(num_calibration_samples):
        # Yield single sample
        yield [X_train_final[i:i+1].astype(np.float32)]

print('Creating INT8 quantized model...')
print(f'Using {min(500, len(X_train_final))} calibration samples')

# Create converter
converter = tf.lite.TFLiteConverter.from_keras_model(model)

# Set optimization flags for INT8 quantization
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset_gen

# Ensure all ops are INT8
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

# Convert
print('Converting... (this may take a minute)')
tflite_model_int8 = converter.convert()

# Save
tflite_int8_path = TFLITE_DIR / f'model_{feature_type}_int8.tflite'
with open(tflite_int8_path, 'wb') as f:
    f.write(tflite_model_int8)

tflite_int8_size_mb = len(tflite_model_int8) / (1024 * 1024)
tflite_int8_size_kb = len(tflite_model_int8) / 1024

print(f'\n✓ TFLite INT8 model saved: {tflite_int8_path.name}')
print(f'  Size: {tflite_int8_size_mb:.2f} MB ({tflite_int8_size_kb:.0f} KB)')
print(f'  Compression vs FP32: {(1 - tflite_int8_size_mb/keras_size_mb) * 100:.1f}%')
print(f'  Compression vs TFLite FP32: {(1 - tflite_int8_size_mb/tflite_fp32_size_mb) * 100:.1f}%')

# Check if meets ESP32-S3 target
target_kb = 500
if tflite_int8_size_kb < target_kb:
    print(f'\n✓ Model meets ESP32-S3 target (<{target_kb}KB)!')
else:
    print(f'\n⚠️  Model exceeds target: {tflite_int8_size_kb - target_kb:.0f} KB over')
    print('  Consider: reducing model complexity, pruning, or using 1ch instead of 3ch')

print('='*70)

## Section 8: Evaluate INT8 Quantized Model

In [ ]:
print('\nEVALUATING INT8 QUANTIZED MODEL...')
print('='*70)

# Load INT8 model
interpreter_int8 = tf.lite.Interpreter(model_path=str(tflite_int8_path))
interpreter_int8.allocate_tensors()

# Get input and output details
input_details_int8 = interpreter_int8.get_input_details()
output_details_int8 = interpreter_int8.get_output_details()

print('INT8 model details:')
print(f'  Input dtype:  {input_details_int8[0]["dtype"]}')
print(f'  Output dtype: {output_details_int8[0]["dtype"]}')

# Get quantization parameters
input_scale, input_zero_point = input_details_int8[0]['quantization']
output_scale, output_zero_point = output_details_int8[0]['quantization']

print(f'  Input quantization:  scale={input_scale:.6f}, zero_point={input_zero_point}')
print(f'  Output quantization: scale={output_scale:.6f}, zero_point={output_zero_point}')

# Run inference on test set
print('\nRunning INT8 inference on test set...')
y_pred_int8 = []
inference_times_int8 = []

for i in range(len(X_test_final)):
    # Quantize input
    input_data = X_test_final[i:i+1].astype(np.float32)
    input_data_int8 = (input_data / input_scale + input_zero_point).astype(np.int8)
    
    # Set input tensor
    interpreter_int8.set_tensor(input_details_int8[0]['index'], input_data_int8)
    
    # Run inference
    start = time.time()
    interpreter_int8.invoke()
    inference_times_int8.append((time.time() - start) * 1000)
    
    # Get output and dequantize
    output_data_int8 = interpreter_int8.get_tensor(output_details_int8[0]['index'])
    output_data = (output_data_int8.astype(np.float32) - output_zero_point) * output_scale
    y_pred_int8.append(output_data[0][0])

y_pred_int8 = np.array(y_pred_int8)
y_pred_int8_binary = (y_pred_int8 > 0.5).astype(int)

# Metrics
int8_acc = accuracy_score(y_test, y_pred_int8_binary)
int8_f1 = f1_score(y_test, y_pred_int8_binary)
int8_precision = precision_score(y_test, y_pred_int8_binary)
int8_recall = recall_score(y_test, y_pred_int8_binary)
int8_auc = roc_auc_score(y_test, y_pred_int8)
int8_inference = np.mean(inference_times_int8)

print('\nINT8 Quantized Model:')
print(f'  Accuracy:  {int8_acc:.4f}')
print(f'  F1 Score:  {int8_f1:.4f}')
print(f'  Precision: {int8_precision:.4f}')
print(f'  Recall:    {int8_recall:.4f}')
print(f'  AUC:       {int8_auc:.4f}')
print(f'  FNR:       {(1 - int8_recall) * 100:.2f}%')
print(f'  Inference: {int8_inference:.2f} ms/sample (avg)')

# Compare with FP32
print('\nAccuracy loss due to quantization:')
print(f'  F1:        {(int8_f1 - keras_f1):+.4f} ({(int8_f1 - keras_f1)/keras_f1 * 100:+.2f}%)')
print(f'  Recall:    {(int8_recall - keras_recall):+.4f} ({(int8_recall - keras_recall)/keras_recall * 100:+.2f}%)')
print(f'  Inference: {(int8_inference - keras_inference_time):+.2f} ms')

# Acceptable threshold
acceptable_f1_loss = 0.02  # 2% F1 loss acceptable
f1_loss = keras_f1 - int8_f1

if f1_loss < acceptable_f1_loss:
    print(f'\n✓ Quantization quality: GOOD (F1 loss < {acceptable_f1_loss:.1%})')
else:
    print(f'\n⚠️  Quantization quality: May need tuning (F1 loss = {f1_loss:.4f})')

print('='*70)

## Section 9: Comprehensive Comparison

In [ ]:
print('\nCOMPREHENSIVE MODEL COMPARISON...')
print('='*70)

# Create comparison dataframe
comparison = pd.DataFrame({
    'Model': ['Keras FP32', 'TFLite FP32', 'TFLite INT8'],
    'Accuracy': [keras_acc, tflite_fp32_acc, int8_acc],
    'F1': [keras_f1, tflite_fp32_f1, int8_f1],
    'Precision': [keras_precision, tflite_fp32_precision, int8_precision],
    'Recall': [keras_recall, tflite_fp32_recall, int8_recall],
    'AUC': [keras_auc, tflite_fp32_auc, int8_auc],
    'FNR (%)': [(1-keras_recall)*100, (1-tflite_fp32_recall)*100, (1-int8_recall)*100],
    'Inference (ms)': [keras_inference_time, tflite_fp32_inference, int8_inference],
    'Size (KB)': [keras_size_mb*1024, tflite_fp32_size_mb*1024, tflite_int8_size_kb]
})

print(comparison.to_string(index=False))
print('='*70)

# Save comparison
comparison_csv = RESULTS_DIR / f'08_quantization_comparison_{feature_type}.csv'
comparison.to_csv(comparison_csv, index=False)
print(f'\n✓ Saved comparison: {comparison_csv.name}')

## Section 10: Visualization

In [ ]:
print('\nGenerating visualizations...')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Performance comparison
metrics = ['F1', 'Recall', 'Precision']
keras_vals = [keras_f1, keras_recall, keras_precision]
int8_vals = [int8_f1, int8_recall, int8_precision]

x = np.arange(len(metrics))
width = 0.35

axes[0].bar(x - width/2, keras_vals, width, label='Keras FP32', alpha=0.8)
axes[0].bar(x + width/2, int8_vals, width, label='TFLite INT8', alpha=0.8)
axes[0].set_ylabel('Score')
axes[0].set_title('Performance Comparison', fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics)
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)
axes[0].set_ylim([0.9, 1.0])

# Model size comparison
models = ['Keras\nFP32', 'TFLite\nFP32', 'TFLite\nINT8']
sizes_kb = [keras_size_mb*1024, tflite_fp32_size_mb*1024, tflite_int8_size_kb]
colors = ['steelblue', 'orange', 'green']

bars = axes[1].bar(models, sizes_kb, color=colors, alpha=0.8)
axes[1].axhline(y=500, color='red', linestyle='--', label='ESP32-S3 Target (500KB)', alpha=0.7)
axes[1].set_ylabel('Size (KB)')
axes[1].set_title('Model Size Comparison', fontweight='bold')
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.0f}KB',
                ha='center', va='bottom')

# Inference time comparison
inf_times = [keras_inference_time, tflite_fp32_inference, int8_inference]
bars = axes[2].bar(models, inf_times, color=colors, alpha=0.8)
axes[2].axhline(y=50, color='red', linestyle='--', label='ESP32-S3 Target (<50ms)', alpha=0.7)
axes[2].set_ylabel('Inference Time (ms)')
axes[2].set_title('Inference Speed Comparison', fontweight='bold')
axes[2].legend()
axes[2].grid(axis='y', alpha=0.3)

# Add value labels
for bar in bars:
    height = bar.get_height()
    axes[2].text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.1f}ms',
                ha='center', va='bottom')

plt.tight_layout()
fig_path = FIGURES_DIR / f'08_quantization_comparison_{feature_type}.png'
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
print(f'✓ Saved: {fig_path.name}')
plt.close()

print('✓ Visualization complete')

## Section 11: Deployment Package Generation

In [ ]:
print('\nGENERATING DEPLOYMENT PACKAGE...')
print('='*70)

# Create deployment directory
deploy_dir = PROJECT_ROOT / 'deployment' / f'esp32_s3_{feature_type}'
deploy_dir.mkdir(parents=True, exist_ok=True)

# Copy quantized model
import shutil
deploy_model = deploy_dir / 'model.tflite'
shutil.copy(tflite_int8_path, deploy_model)

print(f'✓ Copied model to: {deploy_model}')

# Create model info JSON
model_info = {
    'model_name': model.name,
    'feature_type': feature_type,
    'input_shape': list(input_details_int8[0]['shape']),
    'output_shape': list(output_details_int8[0]['shape']),
    'input_dtype': str(input_details_int8[0]['dtype']),
    'output_dtype': str(output_details_int8[0]['dtype']),
    'input_quantization': {
        'scale': float(input_scale),
        'zero_point': int(input_zero_point)
    },
    'output_quantization': {
        'scale': float(output_scale),
        'zero_point': int(output_zero_point)
    },
    'model_size_kb': tflite_int8_size_kb,
    'performance': {
        'accuracy': float(int8_acc),
        'f1_score': float(int8_f1),
        'precision': float(int8_precision),
        'recall': float(int8_recall),
        'fnr_percent': float((1-int8_recall)*100),
        'inference_ms_cpu': float(int8_inference)
    },
    'audio_config': {
        'sample_rate': 16000,
        'n_mels': 40,
        'n_fft': 512,
        'hop_length': 160,
        'window_duration_ms': 1000
    },
    'esp32_s3_requirements': {
        'min_flash_mb': 8,
        'estimated_ram_kb': 256,
        'tflite_micro': 'required',
        'i2s_microphone': 'required'
    },
    'deployment_status': 'READY' if tflite_int8_size_kb < 500 else 'SIZE_WARNING',
    'timestamp': datetime.now().isoformat()
}

model_info_path = deploy_dir / 'model_info.json'
with open(model_info_path, 'w') as f:
    json.dump(model_info, f, indent=2)

print(f'✓ Saved model info: {model_info_path.name}')

# Create README
readme_content = f"""# ESP32-S3 Deployment Package

## Model: {model.name} ({feature_type})

### Performance Metrics
- **F1 Score:** {int8_f1:.4f}
- **Recall:** {int8_recall:.4f} (FNR: {(1-int8_recall)*100:.2f}%)
- **Precision:** {int8_precision:.4f}
- **Inference Time:** {int8_inference:.2f} ms/sample (on CPU)

### Model Specifications
- **Size:** {tflite_int8_size_kb:.0f} KB
- **Quantization:** INT8 (full integer)
- **Input:** {input_details_int8[0]['shape']} - {input_details_int8[0]['dtype']}
- **Output:** {output_details_int8[0]['shape']} - {output_details_int8[0]['dtype']}

### Audio Configuration
- **Sample Rate:** 16 kHz
- **Features:** {"Mel + Delta + Delta-Delta (3-channel)" if feature_type == '3ch' else "Mel Spectrogram (1-channel)"}
- **n_mels:** 40
- **Window:** 1 second

### ESP32-S3 Requirements
- **Flash:** Minimum 8 MB
- **RAM:** ~256 KB for inference
- **Libraries:** TensorFlow Lite Micro
- **Hardware:** I2S microphone (e.g., INMP441)

### Deployment Status
{"✅ READY FOR DEPLOYMENT" if tflite_int8_size_kb < 500 else "⚠️ MODEL SIZE EXCEEDS 500KB TARGET"}

### Files
1. `model.tflite` - Quantized INT8 model
2. `model_info.json` - Complete model metadata
3. `README.md` - This file

### Next Steps
1. Integrate with ESP32-S3 firmware (Notebook 09)
2. Implement feature extraction in C/C++
3. Deploy TFLite Micro runtime
4. Test real-time inference on device
5. Optimize power consumption

Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
"""

readme_path = deploy_dir / 'README.md'
with open(readme_path, 'w') as f:
    f.write(readme_content)

print(f'✓ Created README: {readme_path.name}')

print(f'\nDeployment package saved to: {deploy_dir}')
print('\nPackage contents:')
for item in deploy_dir.iterdir():
    size = item.stat().st_size / 1024 if item.is_file() else 0
    print(f'  - {item.name:<30} {size:>8.1f} KB' if size else f'  - {item.name}')

print('='*70)

## Section 12: Final Report & Recommendations

In [ ]:
print('\nFINAL QUANTIZATION REPORT...')
print('='*70)

# Create final report
final_report = {
    'timestamp': datetime.now().isoformat(),
    'notebook': '08_model_quantization_optimization',
    'feature_type': feature_type,
    'source_model': {
        'name': model.name,
        'parameters': int(model.count_params()),
        'size_mb': float(keras_size_mb)
    },
    'quantized_model': {
        'format': 'TensorFlow Lite INT8',
        'size_kb': float(tflite_int8_size_kb),
        'compression_ratio': float(keras_size_mb * 1024 / tflite_int8_size_kb),
        'path': str(tflite_int8_path)
    },
    'performance_metrics': {
        'original': {
            'f1': float(keras_f1),
            'recall': float(keras_recall),
            'precision': float(keras_precision),
            'accuracy': float(keras_acc)
        },
        'quantized': {
            'f1': float(int8_f1),
            'recall': float(int8_recall),
            'precision': float(int8_precision),
            'accuracy': float(int8_acc)
        },
        'degradation': {
            'f1_loss': float(keras_f1 - int8_f1),
            'f1_loss_percent': float((keras_f1 - int8_f1) / keras_f1 * 100),
            'recall_loss': float(keras_recall - int8_recall),
            'acceptable': bool((keras_f1 - int8_f1) < 0.02)
        }
    },
    'esp32_s3_readiness': {
        'size_requirement_met': bool(tflite_int8_size_kb < 500),
        'inference_estimate_ms': float(int8_inference * 2),  # Conservative estimate for ESP32
        'inference_target_met': bool(int8_inference * 2 < 50),
        'deployment_status': 'READY' if (tflite_int8_size_kb < 500 and (keras_f1 - int8_f1) < 0.02) else 'NEEDS_REVIEW'
    },
    'deployment_package': str(deploy_dir),
    'next_steps': [
        'Implement C/C++ feature extraction for ESP32-S3',
        'Port TFLite Micro runtime',
        'Real-time testing on device (Notebook 09)',
        'Power optimization',
        'Field testing'
    ]
}

# Save report
report_path = RESULTS_DIR / f'08_quantization_report_{feature_type}.json'
with open(report_path, 'w') as f:
    json.dump(final_report, f, indent=2)

print(f'✓ Saved final report: {report_path.name}')

# Print summary
print('\n' + '='*70)
print('QUANTIZATION SUMMARY')
print('='*70)

print(f'Model: {model.name} ({feature_type})')
print(f'\nSize Reduction:')
print(f'  Original:  {keras_size_mb:.2f} MB ({keras_size_mb * 1024:.0f} KB)')
print(f'  Quantized: {tflite_int8_size_mb:.2f} MB ({tflite_int8_size_kb:.0f} KB)')
print(f'  Reduction: {(1 - tflite_int8_size_mb/keras_size_mb) * 100:.1f}%')

print(f'Performance:')
print(f'  F1 Score:  {keras_f1:.4f} → {int8_f1:.4f} (loss: {(keras_f1-int8_f1):.4f})')
print(f'  Recall:    {keras_recall:.4f} → {int8_recall:.4f} (loss: {(keras_recall-int8_recall):.4f})')
print(f'  Inference: {keras_inference_time:.2f}ms → {int8_inference:.2f}ms')

print(f'ESP32-S3 Readiness:')
print(f'  Size target (<500KB):      {"✓ PASS" if tflite_int8_size_kb < 500 else "✗ FAIL"} ({tflite_int8_size_kb:.0f} KB)')
print(f'  Inference target (<50ms):  {"✓ LIKELY" if int8_inference < 25 else "⚠️  UNCERTAIN"} (est. {int8_inference*2:.1f}ms on device)')
print(f'  Accuracy acceptable:       {"✓ YES" if (keras_f1 - int8_f1) < 0.02 else "✗ NO"} (F1 loss {(keras_f1-int8_f1):.4f})')

if tflite_int8_size_kb < 500 and (keras_f1 - int8_f1) < 0.02:
    print('✓ READY FOR ESP32-S3 DEPLOYMENT')
elif tflite_int8_size_kb >= 500:
    print('⚠️  WARNING: Model size exceeds target')
    print(f'   Consider: Using 1ch instead of 3ch, or reducing model complexity')
elif (keras_f1 - int8_f1) >= 0.02:
    print('⚠️  WARNING: Significant accuracy loss')
    print(f'   Consider: Quantization-aware training or different model architecture')

print(f'\nDeployment package: {deploy_dir}')

print('='*70)
print('✓ NOTEBOOK 08 COMPLETE')
print('='*70)
print('\nNext: Notebook 09 - ESP32-S3 Integration & Field Testing')
print('='*70)